In [ ]:
!pip install transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12526 sha256=a7fe4306103bf4983173b7cd3955bc7264c293cdf2b9b38d73ad09400fc6b618
  Stored in directory: /root/.cache/pip/wheels/f6/6e/1a/710143d0e50e827ae5b60a47a6d43d715cf1c6e35881a05530
Successfully built transformers-stream-generator


In [ ]:
import torch
import numpy as np
import transformer_lens
from transformer_lens import utilities
import torch.nn as nn
import random
import pickle
import pandas as pd
import gc
from transformer_lens.model_bridge import TransformerBridge
import torch.nn.functional as F
from tqdm.auto import tqdm




In [ ]:
model = TransformerBridge.boot_transformers("Qwen/Qwen3-1.7B",device="cuda")
print("Model device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Model device: cuda:0


In [ ]:
# Matched base/source pairs: only ones digits change; targets are answer tens digits.
import random

template = "{} + {} = {}"

def create_dataset(n_samples=500, seed=0):
    rng = random.Random(seed)
    data, seen = [], set()
    while len(data) < n_samples:
        ah, bh = rng.randint(1, 8), rng.randint(1, 8)
        if ah + bh > 9:
            continue  # The answer's hundreds digit must remain a single digit.
        at, bt = rng.randint(0, 9), rng.randint(0, 9)
        if at + bt > 8:
            continue  # No tens-to-hundreds carry, even after the source ones carry.
        ao, bo = rng.randint(0, 9), rng.randint(0, 9)
        if ao + bo >= 10:
            continue  # Base has no carry from the ones column.
        co, do = rng.randint(1, 9), rng.randint(1, 9)
        if co + do < 10:
            continue  # Source has a carry from the ones column.

        a = 100 * ah + 10 * at + ao
        b = 100 * bh + 10 * bt + bo
        c = 100 * ah + 10 * at + co
        d = 100 * bh + 10 * bt + do
        base_sum, source_sum = a + b, c + d
        if base_sum // 100 != source_sum // 100:
            continue
        if (base_sum // 10) % 10 + 1 != (source_sum // 10) % 10:
            continue

        base_prompt = template.format(a, b, base_sum // 100)
        source_prompt = template.format(c, d, source_sum // 100)
        if len(base_prompt) != len(source_prompt):
            continue
        base_target = (base_sum // 10) % 10
        source_target = (source_sum // 10) % 10
        pair = (base_prompt, source_prompt, base_target, source_target)
        if pair not in seen:
            data.append(pair)
            seen.add(pair)
    return data


In [ ]:
# Preview raw candidate pairs. The final two values are base/source tens-digit targets.
for base_prompt, source_prompt, base_target, source_target in create_dataset(6, seed=0):
    print(f"base={base_prompt!r}, source={source_prompt!r}, targets=({base_target}, {source_target})")


base='541 + 315 = 8', source='548 + 319 = 8', targets=(5, 6)
base='110 + 169 = 2', source='118 + 166 = 2', targets=(7, 8)
base='422 + 530 = 9', source='425 + 538 = 9', targets=(5, 6)
base='220 + 221 = 4', source='229 + 227 = 4', targets=(4, 5)
base='713 + 120 = 8', source='719 + 122 = 8', targets=(3, 4)
base='211 + 534 = 7', source='216 + 537 = 7', targets=(4, 5)


In [ ]:
# Check token positions. The answer's hundreds digit is included; the next token is its tens digit.
sample_prompt = "410 + 432 = 8"
print(sample_prompt)
tokens = model.to_tokens(sample_prompt)
model.to_str_tokens(sample_prompt)


410 + 432 = 8


['4', '1', '0', ' +', ' ', '4', '3', '2', ' =', ' ', '8']

In [ ]:
#select nodes: skip first four tokens and skip last 20/30% layers
# NOTE: the ones digit of the 2nd number is at position 7, so the ones carry cannot exist before position 7
# (causal attention). Positions 4-6 are therefore never expected to be selected -- they act as a built-in
# negative control. Use range(7, 10) if you want a smaller, faster search.
nodes = []
for layer in range(int(0.6*model.cfg.n_layers)):
    for pos in range(5, 11):
        nodes.append((layer, pos))

max_layer = 17
final_prompt_pos = 11

nodes = [
    (layer, pos)
    for layer, pos in nodes
    if layer <= max_layer and pos != final_prompt_pos
]

print(f"Candidate nodes after filtering: {len(nodes)}")
print(nodes)

Candidate nodes after filtering: 96
[(0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (3, 10), (4, 5), (4, 6), (4, 7), (4, 8), (4, 9), (4, 10), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (6, 5), (6, 6), (6, 7), (6, 8), (6, 9), (6, 10), (7, 5), (7, 6), (7, 7), (7, 8), (7, 9), (7, 10), (8, 5), (8, 6), (8, 7), (8, 8), (8, 9), (8, 10), (9, 5), (9, 6), (9, 7), (9, 8), (9, 9), (9, 10), (10, 5), (10, 6), (10, 7), (10, 8), (10, 9), (10, 10), (11, 5), (11, 6), (11, 7), (11, 8), (11, 9), (11, 10), (12, 5), (12, 6), (12, 7), (12, 8), (12, 9), (12, 10), (13, 5), (13, 6), (13, 7), (13, 8), (13, 9), (13, 10), (14, 5), (14, 6), (14, 7), (14, 8), (14, 9), (14, 10), (15, 5), (15, 6), (15, 7), (15, 8), (15, 9), (15, 10)]


In [ ]:
print(len(nodes), "candidate (layer, position) cells; positions", sorted({p for _, p in nodes}))

96 candidate (layer, position) cells; positions [5, 6, 7, 8, 9, 10]


In [ ]:
import time
import random
import torch
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Mask sampling
# ----------------------------------------------------------------------------
def binary_concrete(logits, K=1.0, hard=True):
    """Return a hard forward mask with sigmoid(K * logits) gradients via STE."""
    if K <= 0:
        raise ValueError("K must be positive")
    mask_soft = torch.sigmoid(K * logits)
    if not hard:
        return mask_soft, mask_soft
    mask_hard = (logits > 0).to(logits.dtype)
    mask_used = mask_hard - mask_soft.detach() + mask_soft
    return mask_used, mask_soft


# ----------------------------------------------------------------------------
# Tokenisation / caching
# ----------------------------------------------------------------------------
def _prompt_tokens(model, prompts, device):
    toks = [model.to_tokens(p)[0] for p in prompts]
    L = toks[0].shape[0]
    if any(t.shape[0] != L for t in toks):
        raise ValueError("All prompts must tokenize to the same length")
    return torch.stack(toks).to(device)


def _prepare(model, data, nodes, cfa_tok_id, cache_bs=32):
    """
    Tokenize once and cache SOURCE activations once (on CPU) at only the requested nodes.

    cfa_tok_id : source tens-digit token IDs, shape [N]; one target digit per pair.
    Nodes must lie inside the prompt (pos < L) because source activations come from the
    source prompt alone.
    """
    device = next(model.parameters()).device
    gold = torch.as_tensor(cfa_tok_id, dtype=torch.long)
    if gold.ndim == 1:
        gold = gold[:, None]
    N, K = gold.shape
    if N != len(data) or N == 0:
        raise ValueError("data and cfa_tok_id must have equal, nonzero lengths")

    base_ptok = _prompt_tokens(model, [item[0] for item in data], device)
    src_tok = _prompt_tokens(model, [item[1] for item in data], device)
    L = base_ptok.shape[1]
    if src_tok.shape[1] != L:
        raise ValueError(f"Base ({L}) and source ({src_tok.shape[1]}) prompts differ in token length")

    gold = gold.to(device)
    base_tok = torch.cat([base_ptok, gold[:, : K - 1]], dim=1)  # prompt + teacher-forced answer prefix

    n_layers = model.cfg.n_layers
    layer_entries = {}
    for i, (layer, pos) in enumerate(nodes):
        if not 0 <= pos < L:
            raise ValueError(f"Token position {pos} outside the prompt (length {L})")
        if not 0 <= layer < n_layers:
            raise ValueError(f"Layer {layer} outside model (n_layers={n_layers})")
        layer_entries.setdefault(layer, []).append((i, pos))

    d = model.cfg.d_model
    src_cache = None
    with torch.no_grad():
        for start in range(0, N, cache_bs):
            sl = slice(start, start + cache_bs)
            grabbed = {}

            def make_hook(layer):
                def hook(act, hook=None):
                    grabbed[layer] = torch.stack(
                        [act[:, p, :] for _, p in layer_entries[layer]], dim=0
                    ).detach().cpu()
                    return act
                return hook

            model.run_with_hooks(
                src_tok[sl],
                fwd_hooks=[(f"blocks.{l}.hook_resid_post", make_hook(l)) for l in layer_entries],
                return_type=None,
            )
            if src_cache is None:
                src_cache = torch.zeros(len(nodes), N, d, dtype=next(iter(grabbed.values())).dtype)
            for layer, entries in layer_entries.items():
                for k, (i, _) in enumerate(entries):
                    src_cache[i, sl] = grabbed[layer][k]

    return {"base_tok": base_tok, "src_cache": src_cache, "layer_entries": layer_entries,
            "gold": gold, "L": L, "K": K, "N": N}


def _forward_with_mask(model, prep, idx, m):
    """Interchange-intervened forward on base examples `idx`; m is a [n_nodes] 0/1 (or relaxed) tensor.
    Returns logits at the K answer positions: [B, K, vocab]."""
    base_tok, layer_entries = prep["base_tok"], prep["layer_entries"]
    device = base_tok.device
    src_b = prep["src_cache"][:, idx].to(device)  # [n_nodes, B, d]

    def make_hook(layer):
        entries = layer_entries[layer]

        def hook(act, hook=None):
            act = act.clone()
            for i, p in entries:
                cur = act[:, p, :]
                act[:, p, :] = cur + m[i].to(act.dtype) * (src_b[i].to(act.dtype) - cur)
            return act
        return hook

    logits = model.run_with_hooks(
        base_tok[idx.to(device)],
        fwd_hooks=[(f"blocks.{l}.hook_resid_post", make_hook(l)) for l in layer_entries],
        return_type="logits",
    )
    L, K = prep["L"], prep["K"]
    return logits[:, L - 1: L - 1 + K, :]


# ----------------------------------------------------------------------------
# Evaluation helpers
# ----------------------------------------------------------------------------
@torch.no_grad()
def evaluate_iia(model, data, nodes, gold_ids, mask_hard, batch_size=16, prepared=None, flip=None):
    """
    Exact IIA with a fixed hard mask (no noise).
    IIA = fraction of base prompts where the intervened next token matches the source tens digit.
    `flip` optionally separates carry-changing from carry-preserving examples.
    """
    model.eval()
    prep = prepared if prepared is not None else _prepare(model, data, nodes, gold_ids)
    device = prep["base_tok"].device
    m = torch.as_tensor(mask_hard).to(device).float()
    N, K = prep["N"], prep["K"]
    hit = torch.zeros(N, K, dtype=torch.bool)
    for s in range(0, N, batch_size):
        idx = torch.arange(s, min(s + batch_size, N))
        pred = _forward_with_mask(model, prep, idx, m).argmax(-1).cpu()
        hit[idx] = pred == prep["gold"][idx.to(device)].cpu()
    ok = hit.all(-1)
    out = {"iia": ok.float().mean().item(), "per_digit": hit.float().mean(0).tolist()}
    if flip is not None:
        f = torch.as_tensor(flip, dtype=torch.bool)
        out["iia_flip"] = ok[f].float().mean().item() if f.any() else float("nan")
        out["iia_same"] = ok[~f].float().mean().item() if (~f).any() else float("nan")
    return out


def random_mask_baseline(model, data, nodes, gold_ids, k, n_trials=10, prepared=None, flip=None):
    """Mean IIA of random masks with k active nodes -- the reference the found mask must beat."""
    prep = prepared if prepared is not None else _prepare(model, data, nodes, gold_ids)
    runs = []
    for _ in range(n_trials):
        m = torch.zeros(len(nodes))
        m[torch.randperm(len(nodes))[:k]] = 1
        runs.append(evaluate_iia(model, data, nodes, gold_ids, m, prepared=prep, flip=flip))
    return {key: float(sum(r[key] for r in runs) / len(runs)) for key in runs[0] if key != "per_digit"}


def leave_one_out(model, data, nodes, gold_ids, mask_hard, prepared=None, flip=None):
    """IIA after removing each selected node in turn. A big drop means the node is necessary."""
    prep = prepared if prepared is not None else _prepare(model, data, nodes, gold_ids)
    mask_hard = torch.as_tensor(mask_hard).clone()
    rows = []
    for i in torch.nonzero(mask_hard).flatten().tolist():
        m = mask_hard.clone()
        m[i] = 0
        r = evaluate_iia(model, data, nodes, gold_ids, m, prepared=prep, flip=flip)
        rows.append((nodes[i], r.get("iia_flip", r["iia"])))
    return rows


def summarize_nodes(selected_nodes):
    """Group selected (layer, pos) cells by token position and show layer ranges."""
    by_pos = {}
    for layer, pos in selected_nodes:
        by_pos.setdefault(pos, []).append(layer)
    lines = []
    for pos in sorted(by_pos):
        layers = sorted(by_pos[pos])
        runs, start = [], layers[0]
        for a, b in zip(layers, layers[1:] + [None]):
            if b is None or b != a + 1:
                runs.append(f"{start}" if start == a else f"{start}-{a}")
                start = b
        lines.append(f"  pos {pos}: layers {', '.join(runs)}")
    return "\n".join(lines) if lines else "  (no nodes selected)"


# ----------------------------------------------------------------------------
# The alignment search
# ----------------------------------------------------------------------------
def alignment_find(
    data, model, nodes, cfa_tok_id,
    alpha_init=None, init_logit=-3.0,
    lambda_=0.1, K=1.0, lr=5e-2, n_epochs=100,
    micro_batch=8,
    penalty_kind="sigmoid", hard=True, verbose=True,
    prepared=None, seed=None, log_every=25,
):
    """
    data        : list of (base_prompt, source_prompt, base_tens, source_tens), equal token length
    nodes       : list of unique (layer, token_pos) cells (resid_post, pos inside the prompt)
    cfa_tok_id  : source tens-digit token IDs, [N]
    alpha_init  : optional tensor [n_nodes] of initial selection logits

    Each epoch uses the entire dataset. Source activations are cached once on CPU; gradients accumulate
    over micro-batches, then flow through the straight-through mask estimator.
    """
    if not nodes or len(set(nodes)) != len(nodes):
        raise ValueError("nodes must be non-empty and unique")
    if penalty_kind not in ("l1", "sigmoid"):
        raise ValueError("penalty_kind must be 'l1' or 'sigmoid'")
    if seed is not None:
        torch.manual_seed(seed)

    model.eval()
    model.requires_grad_(False)
    device = next(model.parameters()).device
    t0 = time.time()

    prep = prepared if prepared is not None else _prepare(model, data, nodes, cfa_tok_id)
    gold, N, num_targets = prep["gold"], prep["N"], prep["K"]
    if verbose:
        print(f"[prep] cached sources for {len(nodes)} nodes x {N} pairs in {time.time() - t0:.1f}s")

    if alpha_init is None:
        alpha = torch.full((len(nodes),), float(init_logit), device=device)
    else:
        alpha = alpha_init.detach().to(device).float().clone()
    alpha = torch.nn.Parameter(alpha)
    opt = torch.optim.Adam([alpha], lr=lr)

    for epoch in range(n_epochs):
        opt.zero_grad(set_to_none=True)
        idx_all = torch.arange(N)

        mask_used, mask_soft = binary_concrete(alpha, K=K, hard=hard)
        m_leaf = mask_used.detach().clone().requires_grad_(True)  # cut the graph; grads accumulate here

        ce_total = 0.0
        for s in range(0, len(idx_all), micro_batch):
            idx = idx_all[s: s + micro_batch]
            with torch.enable_grad():
                logits = _forward_with_mask(model, prep, idx, m_leaf)  # [B, 1, V]
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]).float(),
                    gold[idx.to(device)].reshape(-1),
                    reduction="sum",
                ) / (len(idx_all) * num_targets)
                loss.backward()  # frees this micro-batch graph immediately
            ce_total += loss.item()

        g = m_leaf.grad if m_leaf.grad is not None else torch.zeros_like(m_leaf)
        probs = torch.sigmoid(K * alpha) if penalty_kind == "sigmoid" else mask_soft
        #penalty = lambda_ * probs.mean()
        probs = torch.sigmoid(K * alpha)
        penalty = lambda_ * probs.sum()
        # One backward handles the mask objective and sparsity penalty together.
        ((mask_used * g).sum() + penalty).backward()
        opt.step()

        if verbose and (epoch % log_every == 0 or epoch == n_epochs - 1):
            with torch.no_grad():
                sel = int((alpha > 0).sum().item())
            print(f"[epoch {epoch + 1:4d}/{n_epochs}] ce={ce_total:.4f} pen={penalty.item():.4f} "
                  f"selected={sel}/{len(nodes)}  ({time.time() - t0:.0f}s)")
            print(f"alpha: min={alpha.min().item():.4f}, max={alpha.max().item():.4f}, "
                    f"mean={alpha.mean().item():.4f}")
            print(f"grad_norm={alpha.grad.norm().item():.6f}")

    with torch.no_grad():
        hard_mask = (alpha > 0).to(torch.int64).cpu()
    return {
        "alpha": alpha.detach().cpu(),
        "mask_hard": hard_mask,
        "selected_nodes": [nodes[i] for i in range(len(nodes)) if hard_mask[i].item() == 1],
        "seconds": time.time() - t0,
    }

In [ ]:
# -----------------------------------------------------------------------------
# Model-side helpers and preliminary baseline checks
# -----------------------------------------------------------------------------
def get_digit_ids(model):
    ids = []
    for i in range(10):
        t = model.to_tokens(str(i))
        if t.shape[1] != 1:
            raise ValueError(f"digit {i} tokenizes to {t.shape[1]} tokens (BOS added?)")
        ids.append(t[0, -1].item())
    return ids


def check_tokenization(model, prompt="410 + 432 = 8"):
    toks = model.to_str_tokens(prompt)
    expected_prefix = ['4', '1', '0', ' +', ' ', '4', '3', '2', ' =', ' ']
    if list(toks[:10]) != expected_prefix or len(toks) != 11:
        raise ValueError(f"Unexpected tokenization {list(toks)}; node positions 7-9 assume the first 10 tokens "
                         "match the standard three-digit addition prompt.")
    return toks


@torch.no_grad()
def _score_digit_prompts(model, prompts, targets, digit_ids, batch_size=8):
    """Score next-token digit predictions; return per-example correctness and CE."""
    if len(prompts) != len(targets) or not prompts:
        raise ValueError("prompts and targets must have equal nonzero lengths")
    device = next(model.parameters()).device
    all_correct, all_ce = [], []
    digit_ids_t = torch.as_tensor(digit_ids, dtype=torch.long, device=device)
    model.eval()
    for start in range(0, len(prompts), batch_size):
        ps = prompts[start:start + batch_size]
        ys = targets[start:start + batch_size]
        toks = [model.to_tokens(p)[0] for p in ps]
        lengths = {t.shape[0] for t in toks}
        if len(lengths) != 1:
            raise ValueError("Prompts have unequal token lengths; matched prompts must align positionally")
        tokens = torch.stack(toks).to(device)
        logits = model.run_with_hooks(tokens, return_type="logits")[:, -1, :].float()
        target_idx = torch.as_tensor(ys, dtype=torch.long, device=device)
        target_ids = digit_ids_t[target_idx]
        preds = logits.argmax(dim=-1)
        ce = F.cross_entropy(logits, target_ids, reduction="none")
        all_correct.extend((preds == target_ids).cpu().tolist())
        all_ce.extend(ce.cpu().tolist())
    return {
        "correct": all_correct,
        "ce": all_ce,
        "accuracy": float(sum(all_correct) / len(all_correct)),
        "mean_ce": float(sum(all_ce) / len(all_ce)),
    }


def create_filtered_dataset(model, n_samples=300, seed=0, candidate_batch_size=64,
                            max_candidates=50000, eval_batch_size=8):
    """Keep only matched pairs where the unpatched model predicts both base and source tens digits correctly.

    Incorrect pairs are replaced by newly generated candidates. Raw-candidate accuracy/CE are
    reported before filtering; final-dataset accuracy is 100% by construction.
    """
    if n_samples < 1 or candidate_batch_size < 1 or max_candidates < n_samples:
        raise ValueError("Invalid dataset/filter limits")
    digit_ids = get_digit_ids(model)
    filtered, seen = [], set()
    n_scored = 0
    base_correct_total = source_correct_total = 0
    base_ce_total = source_ce_total = 0.0
    kept_base_ce, kept_source_ce = [], []
    batch_index = 0

    while len(filtered) < n_samples and n_scored < max_candidates:
        batch_n = min(candidate_batch_size, max_candidates - n_scored)
        candidates = create_dataset(batch_n, seed=seed + batch_index * 104729)
        batch_index += 1
        candidates = [x for x in candidates if x not in seen]
        if not candidates:
            continue
        seen.update(candidates)
        base_prompts = [x[0] for x in candidates]
        source_prompts = [x[1] for x in candidates]
        base_targets = [x[2] for x in candidates]
        source_targets = [x[3] for x in candidates]
        base_eval = _score_digit_prompts(model, base_prompts, base_targets, digit_ids, eval_batch_size)
        source_eval = _score_digit_prompts(model, source_prompts, source_targets, digit_ids, eval_batch_size)
        n_scored += len(candidates)
        base_correct_total += sum(base_eval["correct"])
        source_correct_total += sum(source_eval["correct"])
        base_ce_total += sum(base_eval["ce"])
        source_ce_total += sum(source_eval["ce"])

        for i, pair in enumerate(candidates):
            if base_eval["correct"][i] and source_eval["correct"][i] and len(filtered) < n_samples:
                filtered.append(pair)
                kept_base_ce.append(base_eval["ce"][i])
                kept_source_ce.append(source_eval["ce"][i])

    print("PRELIMINARY CHECK — unfiltered candidate pool")
    print(f"  Candidate pairs evaluated: {n_scored}")
    if n_scored:
        print(f"  Base prompts:   accuracy={base_correct_total / n_scored:.3f}, CE={base_ce_total / n_scored:.4f}")
        print(f"  Source prompts: accuracy={source_correct_total / n_scored:.3f}, CE={source_ce_total / n_scored:.4f}")
        print(f"  Retained matched pairs: {len(filtered)}/{n_scored} ({len(filtered) / n_scored:.1%}); "
              "a pair is retained only when BOTH predictions are correct")
    if len(filtered) < n_samples:
        raise RuntimeError(f"Only {len(filtered)}/{n_samples} correct matched pairs found after scoring {n_scored} candidates. "
                           "Inspect the raw accuracy/CE above, then increase max_candidates or revisit tokenization/prompt formatting.")
    print("PRELIMINARY CHECK — filtered dataset passed to alignment search")
    print(f"  Base prompts:   accuracy=1.000, CE={sum(kept_base_ce) / len(kept_base_ce):.4f}")
    print(f"  Source prompts: accuracy=1.000, CE={sum(kept_source_ce) / len(kept_source_ce):.4f}")
    return filtered, {
        "candidate_pairs_scored": n_scored,
        "base_candidate_accuracy": base_correct_total / n_scored,
        "base_candidate_ce": base_ce_total / n_scored,
        "source_candidate_accuracy": source_correct_total / n_scored,
        "source_candidate_ce": source_ce_total / n_scored,
        "retained_pairs": len(filtered),
        "filtered_base_ce": sum(kept_base_ce) / len(kept_base_ce),
        "filtered_source_ce": sum(kept_source_ce) / len(kept_source_ce),
    }


def run_carry_search(
    model, nodes, n_samples=300, seed=0, data=None,
    lambda_=0.05, K=1.0, lr=5e-2, n_epochs=100, init_logit=-0.06,
    micro_batch=8, penalty_kind="sigmoid",
    n_random_trials=50, do_leave_one_out=True, verbose=True,
):
    """Search for nodes that transfer the source tens digit into the base prompt."""
    import time
    T_start = time.time()
    random.seed(seed)
    torch.manual_seed(seed)
    model.eval()
    model.requires_grad_(False)

    print("1) checks")
    check_tokenization(model)
    digit_ids_t = torch.as_tensor(get_digit_ids(model), dtype=torch.long)
    print(f"  checked tokenization in {time.time() - T_start:.1f}s")

    screening = None
    if data is None:
        print("2) preliminary prompt check and replacement of incorrect pairs")
        data, screening = create_filtered_dataset(model, n_samples=n_samples, seed=seed)
    else:
        print(f"2) using {len(data)} pre-screened matched pairs")
    labels = torch.tensor([item[3] for item in data], dtype=torch.long)
    if any(label < 0 or label > 9 for label in labels.tolist()):
        raise ValueError("Source tens-digit targets must be single digits")
    gold_ids = digit_ids_t[labels]
    print(f"  {len(data)} matched pairs; base has no ones carry, source has a ones carry")
    print(f"  first base/source tens targets: {[(item[2], item[3]) for item in data[:8]]}")

    print("3) caching source activations")
    prep = _prepare(model, data, nodes, gold_ids)
    print(f"  cached in {time.time() - T_start:.1f}s")

    print("4) empty-mask baseline")
    zero = evaluate_iia(model, data, nodes, gold_ids, torch.zeros(len(nodes)), prepared=prep)
    print(f"  no intervention: source-tens accuracy={zero['iia']:.3f}")

    print("5) alignment search")
    res = alignment_find(
        data, model, nodes, gold_ids, init_logit=init_logit,
        lambda_=lambda_, K=K, lr=lr, n_epochs=n_epochs,
        micro_batch=micro_batch, penalty_kind=penalty_kind,
        verbose=verbose, seed=seed, prepared=prep,
    )

    print("6) evaluation on the same dataset used for optimization")
    k = int(res["mask_hard"].sum())
    found = evaluate_iia(model, data, nodes, gold_ids, res["mask_hard"], prepared=prep)
    print(f"  selected {k}/{len(nodes)} nodes")
    print(summarize_nodes(res["selected_nodes"]))
    print(f"  FOUND mask: source-tens accuracy={found['iia']:.3f}; target-digit accuracy={found['per_digit'][0]:.3f}")
    rand = None
    if k > 0:
        rand = random_mask_baseline(model, data, nodes, gold_ids, k,
                                    n_trials=n_random_trials, prepared=prep)
        print(f"  RANDOM mask of same size (mean of {n_random_trials}): source-tens accuracy={rand['iia']:.3f}")
    else:
        print("  no nodes selected -> lower lambda_ or inspect optimization")

    loo = None
    if do_leave_one_out and 0 < k <= 40:
        print("7) leave-one-out accuracy after removing each selected node")
        loo = leave_one_out(model, data, nodes, gold_ids, res["mask_hard"], prepared=prep)
        for node, v in sorted(loo, key=lambda x: x[1]):
            print(f"  without {node}: {v:.3f}")

    print(f"done in {time.time() - T_start:.1f}s")
    return {"result": res, "metrics": {"found": found, "random": rand, "empty": zero},
            "leave_one_out": loo, "data": data, "gold_ids": gold_ids, "screening": screening}


In [ ]:
# Screen once; all search runs below reuse the same correctly answered matched pairs.
screened_data, baseline_metrics = create_filtered_dataset(
    model, n_samples=5000, seed=0, candidate_batch_size=64, max_candidates=50000, eval_batch_size=8
)


PRELIMINARY CHECK — unfiltered candidate pool
  Candidate pairs evaluated: 384
  Base prompts:   accuracy=0.938, CE=0.2832
  Source prompts: accuracy=0.875, CE=0.5287
  Retained matched pairs: 300/384 (78.1%); a pair is retained only when BOTH predictions are correct
PRELIMINARY CHECK — filtered dataset passed to alignment search
  Base prompts:   accuracy=1.000, CE=0.0640
  Source prompts: accuracy=1.000, CE=0.0831


In [ ]:
out = run_carry_search(
    model, nodes,
    data=screened_data,
    n_samples=325,
    lambda_=0.0075,
    K=4.0,              # sigmoid slope: sigmoid(K * logits)
    n_epochs=200,
    lr=5e-3,
    micro_batch=8,       # lower if you hit CUDA out-of-memory
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes


1) checks
  checked tokenization in 0.1s
2) using 300 pre-screened matched pairs
  300 matched pairs; base has no ones carry, source has a ones carry
  first base/source tens targets: [(5, 6), (7, 8), (5, 6), (3, 4), (4, 5), (8, 9), (8, 9), (4, 5)]
3) caching source activations
  cached in 3.8s
4) empty-mask baseline
  no intervention: source-tens accuracy=0.000
5) alignment search
[prep] cached sources for 96 nodes x 300 pairs in 0.0s
[epoch    1/200] ce=5.6429 pen=0.3170 selected=0/96  (10s)
alpha: min=-0.0650, max=-0.0550, mean=-0.0616
grad_norm=0.906898
[epoch   26/200] ce=2.5165 pen=0.2814 selected=24/96  (260s)
alpha: min=-0.1974, max=0.0361, mean=-0.1173
grad_norm=0.281893
[epoch   51/200] ce=2.5035 pen=0.2316 selected=15/96  (517s)
alpha: min=-0.3357, max=0.1549, mean=-0.2000
grad_norm=0.281558
[epoch   76/200] ce=2.5035 pen=0.1913 selected=5/96  (773s)
alpha: min=-0.4538, max=0.3014, mean=-0.2784
grad_norm=0.239630
[epoch  101/200] ce=2.5035 pen=0.1619 selected=4/96  (1032s)
a

[(3, 7), (14, 8), (15, 9), (15, 10)]

In [ ]:
out = run_carry_search(
    model, nodes,
    data=screened_data,
    n_samples=325,
    lambda_=0.009,
    K=4.0,              # sigmoid slope: sigmoid(K * logits)
    n_epochs=250,
    lr=5e-3,
    micro_batch=8,       # lower if you hit CUDA out-of-memory
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes


1) checks
  checked tokenization in 0.1s
2) using 300 pre-screened matched pairs
  300 matched pairs; base has no ones carry, source has a ones carry
  first base/source tens targets: [(5, 6), (7, 8), (5, 6), (3, 4), (4, 5), (8, 9), (8, 9), (4, 5)]
3) caching source activations
  cached in 4.4s
4) empty-mask baseline
  no intervention: source-tens accuracy=0.000
5) alignment search
[prep] cached sources for 96 nodes x 300 pairs in 0.0s
